In [ ]:
%matplotlib widget

import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output
from mpl_toolkits.axes_grid1 import make_axes_locatable
import os
import torch
import nibabel as nib
import pydicom

In [ ]:
def load_ct_dicom(volume_path):
    dicom_file_paths = [x for x in os.listdir(dicom_folder_path) if x.endswith(".dcm")]

    dcms = [
        pydicom.dcmread(os.path.join(dicom_folder_path, x), stop_before_pixels=True)
        for x in dicom_file_paths
    ]

    series_ids = set([x.SeriesInstanceUID for x in dcms])
    assert len(series_ids), f"Found multiple series ids: {series_ids}"

    if all(hasattr(x, "ImagePositionPatient") for x in dcms):
        dcms.sort(key=lambda x: x.ImagePositionPatient[2])
    elif all(hasattr(x, "InstanceNumber") for x in dcms):
        dcms.sort(key=lambda x: x.InstanceNumber)
    else:
        raise AttributeError(
            "Dicom series did not have either ImagePositionPatient or InstanceNumber. Can't sort. "
        )

    image_array = np.stack(
        [dcm.pixel_array.astype("float32", copy=False) for dcm in dcms]
    )

    slope = dcms[0].RescaleSlope
    intercept = dcms[0].RescaleIntercept
    image_array = image_array * slope + intercept

    image_array = np.clip(image_array, -1000, 1900)

    return image_array


def load_ct_nifti(volume_path):
    img = nib.load(volume_path)
    hdr = img.header
    slope = float(hdr.get("scl_slope", 1.0)) or 1.0
    inter = float(hdr.get("scl_inter", 0.0)) or 0.0

    print(slope, inter)

    image_array = img.get_fdata()
    image_array = np.clip(image_array, -1000, 1900)

    return image_array


def load_ct_volume(volume_path: str):
    if volume_path.endswith((".nii", ".nii.gz")):
        return load_ct_nifti(volume_path)

    if volume_path.endswith((".pth", ".npy")):
        return np.load(volume_path)

    if os.path.isdir(volume_path):
        if any(x.endswith(".dcm") for x in os.listdir(volume_path)):
            return load_ct_dicom(volume_path)

    raise RuntimeError(f"Failed to load.")

In [ ]:
def view_ct_volume(volume, vmin=None, vmax=None, axis=0, rot=0):

    if axis == 0:
        data = volume  # (Z, Y, X)
    elif axis == 1:
        data = np.transpose(volume, (1, 0, 2))  # (Y, Z, X)
    elif axis == 2:
        data = np.transpose(volume, (2, 0, 1))  # (X, Z, Y)
    else:
        raise ValueError(f"`axis` must be 0, 1, or 2.")

    data = np.rot90(data, k=rot, axes=(1, 2))

    fig_ratio = data.shape[2] / data.shape[1]
    m = 5
    fig_size = (int(m * fig_ratio), m)

    num_slices = data.shape[0]
    slider = widgets.IntSlider(min=0, max=num_slices - 1, step=1)
    out = widgets.Output()

    def update(change):
        with out:
            clear_output(wait=True)
            fig, ax = plt.subplots(figsize=fig_size)
            im = ax.imshow(data[change["new"]], cmap="gray", vmin=vmin, vmax=vmax)
            ax.set_title(f"Slice {change['new']} / {num_slices - 1}")
            ax.axis("off")

            divider = make_axes_locatable(ax)
            cax = divider.append_axes("right", size="5%", pad=0.05)
            fig.colorbar(im, cax=cax)

            plt.tight_layout()

            plt.show()

    slider.observe(update, names="value")
    display(slider, out)
    slider.value = num_slices // 2

In [ ]:
volume_path = "/scratch/VM/radio-foundation/datasets-nodicom/AIMI/inspect2/CTPAu/PE4528575/PE4528575.nii"
volume = load_ct_volume(volume_path)
view_ct_volume(
    volume,
    axis=1,
    rot=1,
)